# LangChain Basics — AI Engineering Interview Prep

LCEL chains, prompt templates, memory, and output parsers.

**Install**: `pip install langchain langchain-anthropic`

In [ ]:
import os
from langchain_anthropic import ChatAnthropic
from langchain.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain.schema import HumanMessage, AIMessage, SystemMessage
from langchain.output_parsers import PydanticOutputParser
from langchain.schema.runnable import RunnablePassthrough, RunnableLambda
from pydantic import BaseModel, Field
from typing import List

llm = ChatAnthropic(
    model="claude-haiku-4-5-20251001",
    anthropic_api_key=os.environ.get("ANTHROPIC_API_KEY"),
    max_tokens=500
)

print("LangChain + Claude Haiku ready")

## 1. Prompt Templates

In [ ]:
# Basic ChatPromptTemplate
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a concise ML interviewer. Answers must be under {max_words} words."),
    ("human", "Explain {concept} with a concrete example.")
])

# Invoke directly
messages = prompt.format_messages(max_words=80, concept="dropout")
print("Formatted messages:")
for msg in messages:
    print(f"  [{msg.__class__.__name__}]: {str(msg.content)[:80]}")

response = llm.invoke(messages)
print(f"\nResponse: {response.content}")

## 2. LCEL Chains (LangChain Expression Language)

In [ ]:
from langchain_core.output_parsers import StrOutputParser

# LCEL: chain components with | operator
prompt = ChatPromptTemplate.from_template(
    "Define {term} in machine learning. One sentence only."
)

# Basic chain: prompt | llm | output_parser
chain = prompt | llm | StrOutputParser()

terms = ["precision", "recall", "F1 score", "AUC-ROC"]
for term in terms:
    result = chain.invoke({"term": term})
    print(f"{term}: {result.strip()}")

# Batch invocation
print("\n--- Batch ---")
results = chain.batch([{"term": t} for t in ["gradient descent", "backpropagation"]])
for term, result in zip(["gradient descent", "backpropagation"], results):
    print(f"{term}: {result.strip()}")

## 3. Multi-Step Chain with RunnablePassthrough

In [ ]:
from langchain_core.runnables import RunnableParallel

# Step 1: Identify the ML problem type
classify_prompt = ChatPromptTemplate.from_template(
    "Classify this ML task as: classification, regression, clustering, or generation.\nTask: {task}\nOutput ONLY the type."
)

# Step 2: Recommend an approach based on type
recommend_prompt = ChatPromptTemplate.from_template(
    "Recommend the top 3 algorithms for a {task_type} task. Be specific. Max 60 words."
)

# Chain with passthrough to carry original input
classify_chain = classify_prompt | llm | StrOutputParser()

full_chain = (
    RunnableParallel(task=RunnablePassthrough(), task_type=classify_chain)
    | (lambda x: {"task_type": x["task_type"].strip()})
    | recommend_prompt | llm | StrOutputParser()
)

tasks = [
    "Predict house prices from square footage and location",
    "Group news articles by topic without labels",
]

for task in tasks:
    print(f"Task: {task}")
    result = full_chain.invoke({"task": task})
    print(f"Recommendations: {result.strip()}")
    print("-" * 50)

## 4. Conversation Memory

In [ ]:
# Simple in-memory conversation history
class ConversationChain:
    def __init__(self, system_prompt="You are a helpful ML tutor. Be concise."):
        self.history = []
        self.system_prompt = system_prompt
        self.prompt = ChatPromptTemplate.from_messages([
            ("system", self.system_prompt),
            MessagesPlaceholder(variable_name="history"),
            ("human", "{input}")
        ])
        self.chain = self.prompt | llm | StrOutputParser()

    def chat(self, user_input):
        response = self.chain.invoke({"history": self.history, "input": user_input})
        self.history.append(HumanMessage(content=user_input))
        self.history.append(AIMessage(content=response))
        return response

conv = ConversationChain()
questions = [
    "What is a decision tree?",
    "What are its main disadvantages?",
    "How does Random Forest fix those?",
]

for q in questions:
    print(f"Q: {q}")
    print(f"A: {conv.chat(q).strip()}")
    print()

print(f"History length: {len(conv.history)} messages")

## 5. Structured Output with Pydantic

In [ ]:
class ModelEvaluation(BaseModel):
    model_name: str = Field(description="Name of the ML model")
    strengths: List[str] = Field(description="Top 3 strengths", max_items=3)
    weaknesses: List[str] = Field(description="Top 3 weaknesses", max_items=3)
    best_use_case: str = Field(description="Primary use case")

parser = PydanticOutputParser(pydantic_object=ModelEvaluation)

eval_prompt = ChatPromptTemplate.from_template(
    "Evaluate the {model} algorithm. {format_instructions}"
)

eval_chain = eval_prompt | llm | parser

for model_name in ["XGBoost", "LSTM"]:
    result = eval_chain.invoke({
        "model": model_name,
        "format_instructions": parser.get_format_instructions()
    })
    print(f"\n{result.model_name}")
    print(f"  Strengths: {result.strengths}")
    print(f"  Weaknesses: {result.weaknesses}")
    print(f"  Best for: {result.best_use_case}")

## Interview Tips

- **LCEL `|` operator**: pipes output of one component as input to next. Lazy — nothing runs until `.invoke()`.
- **RunnablePassthrough**: passes input unchanged. Useful for preserving original values in parallel chains.
- **Memory management**: truncate history (keep last N turns) to avoid exceeding context limits.
- **`.batch()` vs `.invoke()`**: batch runs multiple inputs efficiently in parallel.
- **LangChain vs raw API**: use LangChain for complex chains; raw Anthropic SDK for simple use cases.
- **Output parsers**: structured extraction with retry on failure — more robust than manual JSON parsing.